[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/01_%E3%83%87%E3%83%BC%E3%82%BF%E3%81%AE%E7%A8%AE%E9%A1%9E%E3%81%A8%E5%BA%A6%E6%95%B0%E5%88%86%E5%B8%83.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-01. データの種類と度数分布表

**この章で学ぶこと**
- データには種類（質的・量的）があること
- 度数分布表とヒストグラムの作り方・読み方

統計検定4級でよく問われる、いちばん大事な土台です。

## 1. データの種類

| 大分類 | 小分類 | 説明 | 例 |
|---|---|---|---|
| **質的データ** | 名義尺度 | 区別だけ。順番に意味なし | 血液型・クラス |
| | 順序尺度 | 順番に意味あり | 順位・満足度(5段階) |
| **量的データ** | 間隔尺度 | 差に意味。0は便宜的 | 気温(℃)・西暦 |
| | 比例尺度 | 差にも比にも意味。0は無 | 身長・点数・人数 |

> ❓ クイズ：「テストの点数」「好きな教科」「気温」はそれぞれどの尺度？

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 日本語が文字化けするときは次の1行を有効にしてください
# plt.rcParams['font.family'] = 'Hiragino Sans'  # Macの場合
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')
df.head()

各列がどんなデータか確認しましょう。`クラス` は文字列（質的）、`数学` は数値（量的）です。

In [ ]:
df.dtypes

## 2. 度数分布表をつくる

数学の点数を「階級（10点ごとの区間）」に分けて、各区間に何人いるか数えます。
これを **度数** といいます。

In [ ]:
bins = [0, 20, 40, 60, 80, 100]
labels = ['0-19', '20-39', '40-59', '60-79', '80-100']
df['階級'] = pd.cut(df['数学'], bins=bins, right=False, labels=labels)
freq = df['階級'].value_counts().sort_index()
table = pd.DataFrame({'度数': freq})
table['相対度数'] = (table['度数'] / table['度数'].sum()).round(3)
table['累積度数'] = table['度数'].cumsum()
table

**読み取りのポイント**
- 度数：その階級の人数
- 相対度数：全体に対する割合（合計1）
- 累積度数：その階級までの人数の合計

## 3. ヒストグラムで見える化

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df['数学'], bins=bins, edgecolor='white')
plt.xlabel('数学の点数')
plt.ylabel('人数')
plt.title('数学の点数の分布')
plt.show()

> 📊 ヒストグラムは「棒グラフ」と違い、棒どうしがくっつきます（連続した量を表すため）。

---
## 🏆 練習問題（4級スタイル）

**問1.** 次のデータの尺度を答えよ：(a) 都道府県名 (b) マラソンの順位 (c) 体重 (d) 摂氏温度

**問2.** `英語` の点数で、階級 `[0,20,40,60,80,100]` の度数分布表を作ろう。

**問3.** `英語` のヒストグラムを描き、いちばん人数が多い階級を答えよう。

In [ ]:
# 問2・問3


<details><summary>解答例</summary>

問1: (a)名義 (b)順序 (c)比例 (d)間隔

```python
df['英語階級'] = pd.cut(df['英語'], bins=[0,20,40,60,80,100], right=False)
print(df['英語階級'].value_counts().sort_index())
plt.hist(df['英語'], bins=[0,20,40,60,80,100], edgecolor='white'); plt.show()
```
</details>